# vla-hands · Advanced Notebook

**Assumes:** You understand grafts, appendages, and the training loop from `01_intro.ipynb`.

Topics covered:

| Section | Topic |
|---------|-------|
| 2 | **LoRA** — parameter-efficient VLM adaptation (0.5% of params) |
| 3 | **Freezing curriculum** — gradual layer unfreezing for better final performance |
| 4 | **CompositeGraft** — multiple action heads, one VLM forward pass |
| 5 | New multi-appendage environments (FruitCatcher, TreasureHunt, WhackAMole…) |
| 6 | **Auto curriculum** — one-call training for any appendage combination |
| 7 | Vision skip connection deep-dive (TouchscreenAppendage internals) |
| 8 | Saving, loading, HuggingFace Hub upload |

**Runtime:** ~45–90 min on T4 GPU (most cells are fast; full curriculum is optional)

## 1 · Setup

In [ ]:
!pip install -q git+https://github.com/jerod92/project-h.git@claude/vla-robotic-hands-platform-kGiza
# LoRA requires peft:
!pip install -q 'peft>=0.10.0'
print('✅ vla-hands + peft installed')

In [ ]:
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'
processor = AutoProcessor.from_pretrained(MODEL_ID)
vlm = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.float32)

text_cfg = getattr(vlm.config, 'text_config', None)
hidden_dim = (
    text_cfg.hidden_size
    if text_cfg is not None and hasattr(text_cfg, 'hidden_size')
    else vlm.config.hidden_size
)
print(f'VLM loaded: {sum(p.numel() for p in vlm.parameters())/1e6:.0f}M params  hidden_dim={hidden_dim}')

## 2 · LoRA — Parameter-Efficient VLM Adaptation

Standard training only updates the appendage MLP (~50K params). LoRA additionally adapts the VLM's attention layers with tiny rank-decomposition matrices — typically ~0.5% of the VLM, but giving the VLM's representations a chance to specialise for action prediction.

```
W_adapted = W_frozen + (B @ A) * (alpha / r)
```

- `r=8` → ~0.5% trainable; `r=16` → ~1% trainable
- Requires `pip install peft>=0.10.0`
- After training, call `merge_lora(vlm)` to bake weights back in for inference

In [ ]:
from vla_hands.grafting.lora import LoRAConfig, apply_lora, lora_parameter_count, merge_lora

lora_cfg = LoRAConfig(
    r=8,             # rank: higher = more expressive, more memory
    alpha=16.0,      # scaling factor (effective LR multiplier = alpha/r = 2×)
    dropout=0.05,
    target_modules='auto',  # auto-detects q_proj, v_proj for SmolVLM
)

vlm_lora = apply_lora(vlm, lora_cfg)

counts = lora_parameter_count(vlm_lora)
print(f'LoRA applied!')
print(f'  Trainable params:  {counts["trainable"]:,}  ({counts["pct_trainable"]:.2f}% of VLM)')
print(f'  Total VLM params:  {counts["total"]:,}')

In [ ]:
from vla_hands import VLAGraft, GraftConfig, JoystickAppendage, TargetNavEnvironment
from vla_hands import TrainingCurriculum, CurriculumConfig, QUICK_CURRICULUM

# Use LoRA-wrapped VLM — the interface is identical to the original
lora_graft = VLAGraft(
    vlm=vlm_lora,
    appendage=JoystickAppendage(hidden_dim),
    config=GraftConfig(feature_extraction='last'),
)

config = CurriculumConfig(
    bc_steps=300, rl_steps=80,
    device=device,
    save_dir='model_checkpoints/lora_joystick',
    freezing_stages=QUICK_CURRICULUM,
    eval_every=75, log_every=30, eval_episodes=5,
)

env = TargetNavEnvironment(width=224, height=224, max_steps=80)
lora_metrics = TrainingCurriculum(lora_graft, processor, env, config).run()
print('\nLoRA training complete!')

In [ ]:
# After training: merge LoRA weights back into the base model for efficient inference
# (optional — running inference with the PeftModel is equally valid)
vlm_merged = merge_lora(vlm_lora)
print('LoRA weights merged into base VLM.')
print(f'Model type after merge: {type(vlm_merged).__name__}')

## 3 · Freezing Curriculum — Gradual Layer Unfreezing

The default `QUICK_CURRICULUM` keeps the VLM fully frozen — fastest, but the VLM can't adapt.

The `DEFAULT_CURRICULUM` gradually unfreezes VLM layers:

| Stage | VLM layers unfrozen | When |
|-------|---------------------|------|
| appendage_only | 0 | steps 0–499 |
| last_2 | 2 | steps 500–999 |
| last_6 | 6 | steps 1000–1999 |
| full | all | step 2000+ |

This gives the appendage time to learn from the frozen representation before the VLM starts drifting.

In [ ]:
from vla_hands import DEFAULT_CURRICULUM

# Show the curriculum stages
for stage in DEFAULT_CURRICULUM:
    print(f'  step {stage.start_step:4d}+: {stage.name:20s}  layers_unfrozen={stage.n_layers_unfrozen}')

print()
print('Usage in training:')
print('  CurriculumConfig(..., freezing_stages=DEFAULT_CURRICULUM, bc_steps=2500)')

In [ ]:
# Full curriculum training — uncomment to run (~30 min on T4)
#
# from vla_hands import DPadAppendage, GridWorldEnvironment
#
# full_graft = VLAGraft(vlm, DPadAppendage(hidden_dim), GraftConfig())
# config = CurriculumConfig(
#     bc_steps=2500, rl_steps=500,
#     device=device,
#     save_dir='model_checkpoints/full_curriculum_dpad',
#     freezing_stages=DEFAULT_CURRICULUM,
#     eval_every=500, log_every=100,
# )
# metrics = TrainingCurriculum(full_graft, processor, GridWorldEnvironment(), config).run()

print('Full curriculum training cell (commented out to save runtime).')
print('Enable it for the best possible performance on the D-pad grid task.')

## 4 · CompositeGraft — Multiple Action Heads, One Forward Pass

Some tasks require simultaneous control: steer **and** press. Running the VLM twice wastes compute.

`CompositeGraft` solves this: one VLM forward pass fans out to N independent action heads.

```
image + prompt
      ↓
[VLM backbone]  ← one forward pass
      ↓
last-token hidden state
      ├── [JoystickAppendage] → (dx, dy)
      └── [ButtonAppendage]   → press
```

In [ ]:
from vla_hands import ButtonAppendage
from vla_hands.grafting.composite import CompositeGraft

composite = CompositeGraft(
    vlm=vlm,
    appendages={
        'joystick': JoystickAppendage(hidden_dim),
        'button':   ButtonAppendage(hidden_dim),
    },
    config=GraftConfig(feature_extraction='last'),
)

print(f'Appendages: {list(composite.appendages.keys())}')
total_head_params = sum(a.num_parameters() for a in composite.appendages.values())
print(f'Total head params: {total_head_params:,}  (both heads combined)')

# Forward pass returns a dict keyed by appendage name
dummy = {'input_ids': torch.zeros(1, 5, dtype=torch.long)}
with torch.no_grad():
    out = composite(**dummy)

print('\nForward pass output keys:', list(out.keys()))
print(f'  joystick shape: {out["joystick"].shape}  (dx, dy)')
print(f'  button   shape: {out["button"].shape}   (press confidence)')

## 5 · Multi-Appendage Environments

These environments require simultaneous control and are designed for `CompositeGraft`.
They also accept plain single-appendage actions for backwards compatibility.

In [ ]:
from vla_hands.environments.fruit_catcher import FruitCatcherEnvironment
from vla_hands.environments.treasure_hunt import TreasureHuntEnvironment
from vla_hands.environments.whack_a_mole import WhackAMoleEnvironment
from vla_hands.environments.paint_canvas import PaintCanvasEnvironment
from vla_hands.environments.mcq_navigator import MCQNavigatorEnvironment

multi_envs = [
    ('FruitCatcher\n(joystick + button)',    FruitCatcherEnvironment(width=200, height=200)),
    ('TreasureHunt\n(d-pad + button)',       TreasureHuntEnvironment()),
    ('WhackAMole\n(touchscreen)',            WhackAMoleEnvironment(width=200, height=200)),
    ('PaintCanvas\n(touchscreen sequential)',PaintCanvasEnvironment(width=200, height=200)),
    ('MCQNavigator\n(d-pad + multi-button)', MCQNavigatorEnvironment()),
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, env) in zip(axes, multi_envs):
    obs = env.reset(seed=42)
    act = env.expert_action()
    ax.imshow(obs)
    ax.set_title(name, fontsize=9)
    ax.axis('off')
plt.suptitle('Multi-appendage environments', fontsize=13)
plt.tight_layout()
plt.show()

# Show expert actions
for name, env in multi_envs:
    env.reset(seed=1)
    print(f'{name.split(chr(10))[0]:16s}: {env.expert_action()}')

In [ ]:
# Train CompositeGraft on FruitCatcher
# Joint BC: sum of losses from both appendages
from vla_hands.training.trainer import _preprocess

fruit_env = FruitCatcherEnvironment(width=200, height=200, max_steps=60)
composite.to(device)
composite.train()

optimizer = torch.optim.AdamW(
    [p for p in composite.parameters() if p.requires_grad], lr=3e-4
)

BC_STEPS = 400
losses = []
print(f'Training CompositeGraft on FruitCatcher ({BC_STEPS} steps)...')

for step in range(BC_STEPS):
    obs = fruit_env.reset(seed=step)
    expert = fruit_env.expert_action()  # dict: {"joystick": [x, y], "button": 0.0/1.0}

    inputs = _preprocess(processor, obs, fruit_env.prompt, device)
    out = composite(**inputs)

    # Sum losses from both heads
    tgt_joy = torch.tensor([expert['joystick']], dtype=torch.float32, device=device)
    tgt_btn = torch.tensor([[expert['button']]], dtype=torch.float32, device=device)

    loss = (
        composite.appendages['joystick'].action_loss(out['joystick'], tgt_joy)
        + composite.appendages['button'].action_loss(out['button'], tgt_btn)
    )

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(composite.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())

    if (step + 1) % (BC_STEPS // 8) == 0:
        print(f'  step {step+1:4d}/{BC_STEPS}  loss={loss.item():.4f}')

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.title('CompositeGraft BC Loss (joystick + button, summed)')
plt.xlabel('Step'); plt.ylabel('Loss'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
import os

composite.save('model_checkpoints/composite_fruitcatcher')
print('Saved CompositeGraft. Files:')
for f in os.listdir('model_checkpoints/composite_fruitcatcher'):
    size = os.path.getsize(f'model_checkpoints/composite_fruitcatcher/{f}')
    print(f'  {f}  ({size/1024:.1f} KB)')

## 6 · Auto Curriculum — One Call for Any Combination

`auto_curriculum()` automatically:
1. Picks the best environment(s) for your appendage set
2. Builds a `VLAGraft` (single) or `CompositeGraft` (multiple)
3. Runs BC + RL training with a sensible split
4. Returns `(graft, metrics)` ready to use

In [ ]:
from vla_hands.auto import recommended_envs, make_env, make_appendage, make_graft

# Show what environments the auto library recommends for different combos
for combo in [['joystick'], ['dpad'], ['joystick', 'button'], ['dpad', 'multibutton']]:
    recs = recommended_envs(combo)
    env_names = [e for e, _ in recs[:2]]
    print(f'{str(combo):35s} → {env_names}')

In [ ]:
from vla_hands.auto import auto_curriculum

# Single appendage — automatically selects best environment, runs BC+RL
print('Running auto_curriculum for joystick...')
auto_graft, auto_metrics = auto_curriculum(
    vlm=vlm,
    processor=processor,
    appendage_names=['joystick'],
    budget_steps=300,   # total BC+RL steps
    device=device,
    save_dir='model_checkpoints/auto_joystick',
)
print(f'\nResult: {type(auto_graft).__name__}')

In [ ]:
# Multi-appendage — auto builds CompositeGraft
print('Running auto_curriculum for joystick+button...')
composite_auto, composite_metrics = auto_curriculum(
    vlm=vlm,
    processor=processor,
    appendage_names=['joystick', 'button'],
    budget_steps=400,
    device=device,
    save_dir='model_checkpoints/auto_composite',
)
print(f'\nResult: {type(composite_auto).__name__}')
if hasattr(composite_auto, 'appendages'):
    print(f'Appendages: {list(composite_auto.appendages.keys())}')

## 7 · Vision Skip Connection Deep-Dive

The `TouchscreenAppendage` can take a **skip connection** directly from the VLM's vision encoder — bypassing the LLM decoder's information bottleneck.

```
image → [SigLIP/ViT vision encoder]
             │   patch embeddings [B, N_patches, D_vision]
             │   mean-pool → [B, D_vision]  ← vision skip
             │
             └─────────────────────────────────────────┐
image + prompt → [LLM decoder] → last hidden [B, D_llm] │
                                                         ↓
                                          llm_proj(h) ++ vis_proj(v)
                                                         ↓
                                              predictor MLP → (x, y)
```

`VLAGraft` registers a PyTorch forward hook on the vision encoder automatically.

In [ ]:
from vla_hands.appendages.touchscreen import TouchscreenAppendage
from vla_hands.environments.pointing import PointingEnvironment

vision_dim = VLAGraft.detect_vision_dim(vlm)
print(f'Vision encoder dim: {vision_dim}')  # 1152 for SmolVLM SigLIP

# Compare: LLM-only vs. LLM + vision skip
ts_plain  = TouchscreenAppendage(hidden_dim, vision_dim=None)
ts_skip   = TouchscreenAppendage(hidden_dim, vision_dim=vision_dim)

graft_plain = VLAGraft(vlm, ts_plain, GraftConfig())
graft_skip  = VLAGraft(vlm, ts_skip,  GraftConfig())

print(f'\nPlain (LLM only):    {ts_plain.num_parameters():,} params  hook={graft_plain._vision_hook is not None}')
print(f'With vision skip:    {ts_skip.num_parameters():,} params   hook={graft_skip._vision_hook is not None}')
print()
print('The hook captures SigLIP patch embeddings during the VLM forward pass.')
print('They are mean-pooled over spatial patches and fed as skip to the action head.')

In [ ]:
# Train both and compare on the pointing task
from vla_hands import TrainingCurriculum, CurriculumConfig, QUICK_CURRICULUM

pointing_env = PointingEnvironment(width=256, height=256, n_distractors=3, n_colors=5)

def train_pointing(graft, tag, bc_steps=300):
    config = CurriculumConfig(
        bc_steps=bc_steps, rl_steps=0,
        device=device,
        save_dir=f'model_checkpoints/{tag}',
        freezing_stages=QUICK_CURRICULUM,
        eval_every=bc_steps // 3,
        log_every=bc_steps // 10,
        eval_episodes=5,
    )
    return TrainingCurriculum(graft, processor, pointing_env, config).run()

print('Training plain touchscreen (LLM only)...')
metrics_plain = train_pointing(graft_plain, 'ts_plain')

print('\nTraining touchscreen with vision skip...')
metrics_skip  = train_pointing(graft_skip, 'ts_skip')

In [ ]:
from vla_hands import BenchmarkSuite
from vla_hands.utils.viz import plot_training_curves

# Loss comparison
plot_training_curves(
    {'LLM only': metrics_plain, 'LLM + vision skip': metrics_skip},
    title='Pointing Task: LLM-only vs Vision Skip Connection',
    smooth=10,
)

# Benchmark comparison
eval_env = PointingEnvironment(width=256, height=256, n_distractors=3, n_colors=5)
for name, graft in [('Plain (LLM only)', graft_plain), ('With vision skip', graft_skip)]:
    suite = BenchmarkSuite(graft, processor, [eval_env], device=device)
    r = suite.run_benchmark(eval_env, n_episodes=20)
    print(f'{name:22s}: success={r.success_rate:.0%}  reward={r.mean_reward:+.2f}')

## 8 · Saving, Loading, HuggingFace Hub

All graft types save only the appendage weights — the VLM backbone is loaded from HuggingFace Hub separately. Checkpoints are typically 100 KB – 2 MB.

In [ ]:
import os, json

# VLAGraft save/load
graft_skip.save('model_checkpoints/ts_skip_final')
files = os.listdir('model_checkpoints/ts_skip_final')
print('VLAGraft checkpoint:', files)

# CompositeGraft save/load
composite.save('model_checkpoints/composite_final')
files_c = os.listdir('model_checkpoints/composite_final')
print('CompositeGraft checkpoint:', files_c)

# Config inspection
with open('model_checkpoints/ts_skip_final/graft_config.json') as f:
    print('\nGraft config:', json.dumps(json.load(f), indent=2))

In [ ]:
# Reload: from_pretrained() handles VLM loading + appendage restoration in one call
reloaded = VLAGraft.from_pretrained(
    vlm_id=MODEL_ID,
    appendage=TouchscreenAppendage(hidden_dim, vision_dim=vision_dim),
    checkpoint_path='model_checkpoints/ts_skip_final',
    device=device,
)
print(f'Reloaded: {type(reloaded).__name__}')
print(f'Vision hook: {reloaded._vision_hook is not None}')

In [ ]:
# ── Upload to HuggingFace Hub (optional) ──────────────────────────────────────
# pip install -q huggingface_hub
#
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path='model_checkpoints',
#     repo_id='your-username/vla-hands-checkpoints',
#     repo_type='model',
# )
print('Uncomment the block above to push checkpoints to HuggingFace Hub.')

## ✅ Advanced Topics Complete!

You've covered:

- **LoRA** — adapt only 0.5% of VLM params for faster domain specialisation
- **Freezing curriculum** — staged unfreezing gives the appendage time to stabilise
- **CompositeGraft** — N action heads from one VLM forward pass
- **Auto curriculum** — one call for any appendage combination
- **Vision skip connection** — direct spatial features bypass the LLM bottleneck
- **Multi-appendage environments** — FruitCatcher, TreasureHunt, WhackAMole, …

### Ideas for further exploration

```
Architecture ideas:
  • Cross-attention over patch embeddings instead of mean-pool for skip connection
  • Shared projection layer across appendages in CompositeGraft
  • Temporal stack: condition on a sliding window of last-k observations

New appendages:
  • Slider (1D continuous, e.g. zoom level)
  • Gyro (3-axis orientation)
  • Click (touchscreen + button together)

New environments:
  • Snake game (joystick)
  • Simon Says (sequence of button presses)
  • Camera pan/tilt (two joysticks)
  • Robot arm reaching (6-DOF joystick)

Training improvements:
  • Multi-task BC: one graft on 3 environments simultaneously
  • DAgger: interactive data collection from trained policy
  • PPO instead of REINFORCE for more stable RL
```

**Contribute** at the GitHub repo — PRs welcome!

## 9 · Export GIFs

Compare LoRA-trained vs standard, and visualise the CompositeGraft on FruitCatcher.

In [ ]:
from vla_hands.utils.gif import record_expert_gif, save_rollout_gif
from IPython.display import Image as IPImage
import os

os.makedirs('gifs', exist_ok=True)

# LoRA-trained joystick vs expert
record_expert_gif(
    env=TargetNavEnvironment(width=200, height=200),
    path='gifs/expert_targetnav.gif', n_steps=20, seed=7, fps=8,
)
save_rollout_gif(
    graft=lora_graft, processor=processor,
    env=TargetNavEnvironment(width=200, height=200),
    path='gifs/lora_joystick.gif', n_steps=20, seed=42, device=device, fps=6,
)
print('LoRA joystick GIF:')
display(IPImage(filename='gifs/lora_joystick.gif'))

In [ ]:
# CompositeGraft on FruitCatcher — two-head rollout
# save_rollout_gif uses the first action key for the step() call on single-env graft;
# for CompositeGraft we pass the whole action dict directly.
from vla_hands.utils.gif import save_rollout_gif
from vla_hands.environments.fruit_catcher import FruitCatcherEnvironment

save_rollout_gif(
    graft=composite, processor=processor,
    env=FruitCatcherEnvironment(width=200, height=200),
    path='gifs/composite_fruitcatcher.gif',
    n_steps=24, seed=5, device=device, fps=6,
)
print('CompositeGraft FruitCatcher GIF:')
display(IPImage(filename='gifs/composite_fruitcatcher.gif'))

# Vision skip connection — pointing task
save_rollout_gif(
    graft=graft_skip, processor=processor,
    env=PointingEnvironment(width=200, height=200, n_distractors=3),
    path='gifs/touchscreen_skip.gif',
    n_steps=12, seed=3, device=device, fps=4,
)
print('\nTouchscreen (vision skip) pointing GIF:')
display(IPImage(filename='gifs/touchscreen_skip.gif'))